<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting & Cleaning

## Setup

In [38]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import geopandas
from shapely.geometry import shape
import requests
from google.colab import userdata
from google.colab import drive
from pathlib import Path
from pprint import pprint as print

In [12]:
# Mount Google Drive
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [13]:
# Set up the API
data_api = "https://data.cityofnewyork.us/api/v3/views/irx2-vgm4/query.json"

headers = {
    'X-App-Token': userdata.get('token'),
    "Content-Type": "application/json",
}

In [ ]:
# Fetching a sample of data from the right time period
query = """
SELECT
  count(*)
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 1
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
query_data = query_response.json()
print(query_data)

In [ ]:
# Collecting all the taxi data within the time period specified
query = f"""
SELECT
  *
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 12000000
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
print("All Done!!!")
query_data = query_response.json()
taxi_df = pd.DataFrame(query_data)

## Data Structure

In [ ]:
# Preview the first few rows using the dataframe head function
taxi_df.head()

In [ ]:
# Preview the last few rows using the dataframe head function
taxi_df.tail()

In [ ]:
# Displaying the Dataframes shape, columns, and amount of non null values using the info function
taxi_df.info()

In [ ]:
# Drop columns not needed for the project
taxi_df.drop(columns=['request_source', 'store_and_fwd_flag'], axis=1, inplace=True)

In [ ]:
# Converting our two datetime columns to the pandas datetime type
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'], format="mixed")
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'], format="mixed")

## Missing and Duplicate Data

In [ ]:
# Remove duplicate records from the dataset
taxi_df = taxi_df.drop_duplicates().reset_index(drop=True)

In [ ]:
# collecting numeric columns to handle missing values
non_numeric_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'request_source']
numric_columns = [col for col in taxi_df.columns if col not in non_numeric_columns]
for col in numric_columns:
  taxi_df[col] = pd.to_numeric(taxi_df[col], errors='coerce')

In [ ]:
# calculating percentage of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

In [ ]:
# Inspecting the ratecodeid for its values
print(taxi_df['ratecodeid'].value_counts(dropna=False))

In [ ]:
# Fill missing values in ratecodeid
taxi_df['ratecodeid'] = taxi_df['ratecodeid'].fillna(99)

In [ ]:
# Inspecting the congestion_surcharge for its values
print(taxi_df['congestion_surcharge'].value_counts(dropna=False))

In [ ]:
# Handiling congestion surchage values by replacing it with a boolean indicator
taxi_df['congestion_surcharge'].replace(
    {
      2.50: 1,
      0.00: 0,
      -2.50: 1,
      1.00: 1,
      0.75: 1
    },
    inplace=True
)

taxi_df['congestion_surcharge'] = taxi_df['congestion_surcharge'].fillna(-1)

In [ ]:
# Inspecting the airport_fee for its values
print(taxi_df['airport_fee'].value_counts(dropna=False))

In [ ]:
# Handiling airport_fee values by replacing it with a boolean indicator
taxi_df['airport_fee'].replace(
    {
      0.00: 0,
      1.75: 1,
      -1.75: 1,
      6.75: 1,
      5.00: 1,
      0.50: 1
    },
    inplace=True
)

taxi_df['airport_fee'] = taxi_df['airport_fee'].fillna(-1)

In [ ]:
# recheck percentage of missing values in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

## Taxi Zones Data

In [36]:
# Querying for taxi zones data
zones_query = {
    "query":"""
    SELECT
      *
    LIMIT 500
    """
}
zones_response = requests.get(
    "https://data.cityofnewyork.us/api/v3/views/8meu-9t5y/query.json",
    headers=headers,
    params=zones_query
  )
zone_data = zones_response.json()
print("All done!")

'All done!'


In [40]:
zone_df = pd.DataFrame(zone_data)
zone_df["the_geom"] = zone_df["the_geom"].apply(lambda g: shape(g) if isinstance(g, dict) else None)
zone_df = geopandas.GeoDataFrame(zone_df, geometry="the_geom", crs="EPSG:4326")

In [41]:
# Remove metadata columns
zone_df.drop(columns=[':id', ':version', ':created_at', ':updated_at'], axis=1, inplace=True)

In [42]:
# fixing datatypes of three specific columns specified below
zone_df['shape_leng'] = zone_df['shape_leng'].astype("float")
zone_df['shape_area'] = zone_df['shape_area'].astype("float")
zone_df['locationid'] = zone_df['locationid'].astype("int")

In [43]:
# checking our dataframes structure again
zone_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   the_geom    263 non-null    geometry
 1   shape_leng  263 non-null    float64 
 2   shape_area  263 non-null    float64 
 3   zone        263 non-null    object  
 4   locationid  263 non-null    int64   
 5   borough     263 non-null    object  
dtypes: float64(2), geometry(1), int64(1), object(2)
memory usage: 12.5+ KB


## Export

In [46]:
# Export the processed taxi records and spatial zone lookup tables to Google Drive
taxi_df.to_csv("/content/drive/MyDrive/taxi_summer_data.csv", index=False)
zone_df.to_file("/content/drive/MyDrive/taxi_zones_data.json", index=False)